# Manejo del sistema de ficheros distribuido con `hdfs dfs`

En este cuaderno vas a practicar los comandos más importantes para trabajar con HDFS (Hadoop Distributed File System). Como en el cuaderno anterior, puedes ejecutar comandos del sistema operativo desde Jupyter: con `!` al inicio de una línea, o con `%%bash` al principio de un bloque de varias líneas.

Cada sección contiene una breve explicación, uno o varios ejemplos ya resueltos que debes ejecutar y observar, y un ejercicio con un bloque vacío para que lo completes tú. Los ejercicios se apoyan en los ejemplos previos, por lo que conviene ejecutar el cuaderno **en orden**.

# 1. Sintaxis general y ayuda

Todos los comandos de manejo de ficheros en HDFS siguen la forma:

`hdfs dfs -<comando> [opciones] <argumentos>`

Algunas ideas importantes antes de empezar:

- Existe un comando equivalente, `hadoop fs`, que funciona igual cuando el sistema de ficheros es HDFS. En este cuaderno usaremos siempre `hdfs dfs`.
- Las rutas pueden ser **absolutas** (empiezan por `/`, por ejemplo `/user/jovyan`) o **relativas**. Las relativas se interpretan a partir del directorio personal del usuario en HDFS (`/user/<usuario>`), no a partir del directorio actual de tu sistema de ficheros local.
- Muchos comandos tienen el mismo nombre y propósito que sus equivalentes POSIX (`ls`, `mkdir`, `cp`, `mv`, `rm`, `cat`, `chmod`...), por lo que te resultarán familiares.
- Hay que distinguir siempre entre el sistema de ficheros **local** (el del nodo) y el **distribuido** (HDFS). Los comandos `-put` y `-get` sirven de puente entre ambos.

La ayuda está siempre a mano. Con `-usage` se obtiene un resumen de la sintaxis y con `-help` la descripción completa de un comando.

In [1]:
%%bash

hdfs dfs -usage ls
echo ------------------------------
hdfs dfs -help mkdir

Usage: hadoop fs [generic options] -ls [-C] [-d] [-h] [-q] [-R] [-t] [-S] [-r] [-u] [-e] [<path> ...]
------------------------------
-mkdir [-p] <path> ... :
  Create a directory in specified location.
                                                  
  -p  Do not fail if the directory already exists 


También puedes averiguar quién eres para HDFS y cuál es tu directorio personal, que es donde se crearán las rutas relativas:

In [2]:
%%bash

hdfs dfs -ls -d .
hdfs getconf -confKey fs.defaultFS

drwxrwxrwx   - root supergroup          0 2026-10-29 11:28 .
hdfs://namenode:8020


**Ejercicio 1:**
- Muestra la ayuda (`-help`) del comando `-put` y averigua qué hacen las opciones `-f` y `-p`.<br>
 `-f` sobreescribe el destino si existe. `-p` Preserva las marcas de tiempo, la propiedad y el modo.
- Muestra la ayuda del comando `-rm` y averigua qué hacen las opciones `-r` y `-skipTrash`.<br>
    `-r` elimina el directorio recursivamente. `-skipTrash` evita la papelera eliminandolo directamente

In [8]:
%%bash
##Realiza el ejercicio en este bloque:
hdfs dfs -help rm

-rm [-f] [-r|-R] [-skipTrash] [-safely] <src> ... :
  Delete all files that match the specified file pattern. Equivalent to the Unix
  command "rm <src>"
                                                                                 
  -f          If the file does not exist, do not display a diagnostic message or 
              modify the exit status to reflect an error.                        
  -[rR]       Recursively deletes directories.                                   
  -skipTrash  option bypasses trash, if enabled, and immediately deletes <src>.  
  -safely     option requires safety confirmation, if enabled, requires          
              confirmation before deleting large directory with more than        
              <hadoop.shell.delete.limit.num.files> files. Delay is expected when
              walking over large directory recursively to count the number of    
              files to be deleted before the confirmation.                       


# 2. Preparación de datos locales

Para practicar necesitamos algunos ficheros en el sistema de ficheros local del nodo. En lugar de descargarlos, vamos a crearlos nosotros con `printf`. Ejecuta el siguiente bloque: crea una carpeta `Datos_locales` con varios ficheros de texto y uno en formato CSV.

In [9]:
%%bash

mkdir -p Datos_locales
printf 'Hola HDFS\nEste es el fichero uno\nTiene tres lineas\n' > Datos_locales/uno.txt
printf 'Segundo fichero de prueba\nHadoop reparte los datos en bloques\nY los replica entre los datanodes\n' > Datos_locales/dos.txt
printf 'Tercer fichero\nMapReduce procesa los datos donde estan\n' > Datos_locales/tres.txt
printf 'extra,1\nextra,2\n' > Datos_locales/extra.txt
printf 'id,nombre,nota\n1,Ana,8\n2,Luis,6\n3,Marta,9\n4,Pablo,7\n' > Datos_locales/notas.csv
ls -lh Datos_locales

total 0
-rw-r--r-- 1 root root 96 Oct 29 12:04 dos.txt
-rw-r--r-- 1 root root 16 Oct 29 12:04 extra.txt
-rw-r--r-- 1 root root 52 Oct 29 12:04 notas.csv
-rw-r--r-- 1 root root 55 Oct 29 12:04 tres.txt
-rw-r--r-- 1 root root 51 Oct 29 12:04 uno.txt


# 3. Listar contenido: `-ls`

`-ls` muestra el contenido de un directorio: permisos, factor de replicación, propietario, grupo, tamaño, fecha de modificación y ruta. Opciones habituales:

- `-h`: tamaños en formato legible (KB, MB, GB).
- `-R`: recursivo, recorre también los subdirectorios.
- `-d`: muestra el propio directorio en lugar de su contenido.
- `-t`: ordena por fecha de modificación.

Observa que la segunda columna de la salida es el factor de replicación (en los directorios aparece un guion, pues la replicación solo tiene sentido para ficheros).

In [11]:
%%bash

echo '--- Raiz del sistema de ficheros distribuido ---'
hdfs dfs -ls /
echo
echo '--- Directorio personal, con tamanos legibles ---'
hdfs dfs -ls -h

--- Raiz del sistema de ficheros distribuido ---
Found 4 items
drwxr-xr-x   - root supergroup          0 2026-10-22 11:40 /system
drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp
drwxr-xr-x   - root supergroup          0 2026-10-22 10:40 /user
drwxrwxrwt   - root root                0 2026-10-22 12:03 /yarn

--- Directorio personal, con tamanos legibles ---
Found 3 items
drwxr-xr-x   - root supergroup          0 2026-10-22 11:44 Biblioteca
drwxr-xr-x   - root supergroup          0 2026-10-29 11:28 Resultados
drwxr-xr-x   - root supergroup          0 2026-10-29 10:41 resultados_Ej6


**Ejercicio 2:**
- Lista el contenido de la raíz (`/`) de manera recursiva y con tamaños legibles. Si hay mucha salida, limítala con `| head -n 30`.
- Muestra únicamente la entrada de tu directorio personal (no su contenido) usando la opción `-d`.

In [15]:
%%bash
##Realiza el ejercicio en este bloque:
hdfs dfs -ls -h | head -n 30

Found 3 items
drwxr-xr-x   - root supergroup          0 2026-10-22 11:44 Biblioteca
drwxr-xr-x   - root supergroup          0 2026-10-29 11:28 Resultados
drwxr-xr-x   - root supergroup          0 2026-10-29 10:41 resultados_Ej6


# 4. Crear directorios y ficheros vacíos: `-mkdir` y `-touchz`

`-mkdir` crea directorios. Sin opciones, falla si el directorio padre no existe; con `-p` crea toda la ruta intermedia y no da error si ya existe (igual que en Linux). Por eso es la opción recomendada, sobre todo si vas a ejecutar el bloque varias veces.

`-touchz` crea un fichero de tamaño cero. Ojo: a diferencia del `touch` de Linux, no actualiza la fecha de un fichero existente y falla si el fichero ya existe y no está vacío.

In [ ]:
%%bash

hdfs dfs -mkdir -p Practicas_HDFS/Textos
hdfs dfs -mkdir -p Practicas_HDFS/Datos/2026
hdfs dfs -touchz Practicas_HDFS/vacio.txt
hdfs dfs -ls -R Practicas_HDFS

**Ejercicio 3:** crea, en una sola orden, la estructura de directorios `Proyecto/entrada`, `Proyecto/salida` y `Proyecto/temporal` dentro de tu directorio personal de HDFS. Pista: puedes usar la expansión de llaves de bash, `Proyecto/{entrada,salida,temporal}`. Después, comprueba el resultado con `-ls -R`.

In [16]:
%%bash
##Realiza el ejercicio en este bloque:
hdfs dfs -mkdir -p Proyecto/{entrada,salida,temporal}

hdfs dfs -ls -R

drwxr-xr-x   - root supergroup          0 2026-10-22 11:44 Biblioteca
drwxr-xr-x   - root supergroup          0 2026-10-22 11:44 Biblioteca/Cervantes
-rw-r--r--   3 root supergroup    2225803 2026-10-22 11:19 Biblioteca/Cervantes/Don_Quijote.txt
-rw-r--r--   3 root supergroup     257524 2026-10-22 11:38 Biblioteca/Cervantes/entremeses.txt
-rw-r--r--   3 root supergroup    1056961 2026-10-22 11:38 Biblioteca/Cervantes/novelas_ejemplares.txt
-rw-r--r--   3 root supergroup     345468 2026-10-22 11:38 Biblioteca/Cervantes/pamaso.txt
drwxr-xr-x   - root supergroup          0 2026-10-29 12:08 Proyecto
drwxr-xr-x   - root supergroup          0 2026-10-29 12:08 Proyecto/entrada
drwxr-xr-x   - root supergroup          0 2026-10-29 12:08 Proyecto/salida
drwxr-xr-x   - root supergroup          0 2026-10-29 12:08 Proyecto/temporal
drwxr-xr-x   - root supergroup          0 2026-10-29 11:28 Resultados
drwxr-xr-x   - root supergroup          0 2026-10-29 11:00 Resultados/Cijo_Quijote
-rw-r--r--   3 r

# 5. Subir datos a HDFS: `-put`, `-copyFromLocal`, `-moveFromLocal` y `-appendToFile`

Para llevar ficheros del sistema local al distribuido disponemos de varias opciones:

- `-put`: copia uno o varios ficheros locales al destino indicado. No sobrescribe si el fichero ya existe, salvo que se use `-f`.
- `-copyFromLocal`: prácticamente idéntico a `-put`, pero restringido a orígenes locales.
- `-moveFromLocal`: igual que `-put`, pero borra el fichero local tras subirlo.
- `-appendToFile`: añade el contenido de uno o varios ficheros locales al final de un fichero que ya está en HDFS. Los ficheros de HDFS no se pueden modificar en medio (escribir una vez, leer muchas), pero sí se les puede añadir información al final.

In [ ]:
%%bash

# Subida de un fichero
hdfs dfs -put -f Datos_locales/uno.txt Practicas_HDFS/Textos

# Subida de varios ficheros a la vez (el ultimo argumento es siempre el destino)
hdfs dfs -put -f Datos_locales/dos.txt Datos_locales/tres.txt Practicas_HDFS/Textos

# Subida con copyFromLocal
hdfs dfs -copyFromLocal -f Datos_locales/notas.csv Practicas_HDFS/Datos

hdfs dfs -ls -R Practicas_HDFS

Veamos ahora la diferencia entre subir y mover. Crearemos un fichero local temporal, lo moveremos a HDFS, y comprobaremos que ha desaparecido del sistema local pero existe en el distribuido.

In [ ]:
%%bash

echo 'Fichero que sera movido' > Datos_locales/movido.txt
hdfs dfs -moveFromLocal Datos_locales/movido.txt Practicas_HDFS/Datos

echo '--- Local (ya no deberia estar movido.txt) ---'
ls Datos_locales
echo '--- HDFS ---'
hdfs dfs -ls Practicas_HDFS/Datos

Por último, `-appendToFile` añade al final de un fichero existente. Más adelante veremos cómo comprobar el resultado con `-cat`.

In [ ]:
%%bash

hdfs dfs -appendToFile Datos_locales/extra.txt Practicas_HDFS/Textos/uno.txt
hdfs dfs -ls Practicas_HDFS/Textos

**Ejercicio 4:**
- Crea un fichero local llamado `Datos_locales/ejercicio4.txt` con el texto que quieras (por ejemplo, con `echo`).
- Súbelo a `Practicas_HDFS/Textos` usando `-put`.
- Intenta subirlo de nuevo con `-put` sin la opción `-f` y observa el mensaje de error. ¿Qué dice?
- Añade a ese fichero en HDFS el contenido de `Datos_locales/extra.txt` con `-appendToFile`.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 6. Ver el contenido de los ficheros: `-cat`, `-head`, `-tail` y `-text`

- `-cat`: muestra el contenido completo de uno o varios ficheros. Con ficheros grandes conviene limitar la salida con una tubería (`| head`).
- `-head`: muestra el primer kilobyte del fichero.
- `-tail`: muestra el último kilobyte del fichero (con `-f` sigue mostrando lo que se añade, como `tail -f`).
- `-text`: como `-cat`, pero además decodifica formatos binarios habituales en Hadoop (SequenceFile, Avro) y ficheros comprimidos como `.gz`.

Todos ellos admiten comodines (`*`, `?`), que interpreta el propio HDFS.

In [ ]:
%%bash

echo '--- cat de un fichero (observa lo que anadimos con appendToFile) ---'
hdfs dfs -cat Practicas_HDFS/Textos/uno.txt
echo
echo '--- head ---'
hdfs dfs -head Practicas_HDFS/Textos/dos.txt
echo
echo '--- tail ---'
hdfs dfs -tail Practicas_HDFS/Textos/dos.txt
echo
echo '--- cat con comodin y numeracion de lineas ---'
hdfs dfs -cat 'Practicas_HDFS/Textos/*.txt' | nl

**Ejercicio 5:**
- Muestra el contenido de `notas.csv` (carpeta `Practicas_HDFS/Datos`).
- Muestra solo las dos primeras líneas de `notas.csv`, combinando `-cat` con `head -n 2` mediante una tubería.
- Cuenta cuántas líneas en total tienen todos los ficheros `.txt` de `Practicas_HDFS/Textos` (pista: `-cat` con comodín y `wc -l`).

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 7. Descargar datos de HDFS: `-get`, `-copyToLocal` y `-getmerge`

Es la operación inversa a subir:

- `-get`: copia ficheros o directorios de HDFS al sistema local. Falla si el destino ya existe, salvo que se use `-f`.
- `-copyToLocal`: equivalente a `-get`.
- `-getmerge`: concatena todos los ficheros de un directorio de HDFS en un único fichero local. Es muy útil para recoger el resultado de un MapReduce, que suele estar repartido en varios ficheros `part-r-0000X`.

In [ ]:
%%bash

mkdir -p Descargas_HDFS

# Descarga de un fichero
hdfs dfs -get -f Practicas_HDFS/Textos/uno.txt Descargas_HDFS/

# Descarga de un directorio completo
hdfs dfs -get -f Practicas_HDFS/Datos Descargas_HDFS/

# Descarga fusionando todos los ficheros de un directorio en uno solo
hdfs dfs -getmerge Practicas_HDFS/Textos Descargas_HDFS/todos_los_textos.txt

ls -R Descargas_HDFS
echo
echo '--- Contenido del fichero fusionado ---'
cat Descargas_HDFS/todos_los_textos.txt

**Ejercicio 6:**
- Descarga `notas.csv` a una carpeta local nueva llamada `Descargas_Ejercicio6`, usando `-copyToLocal`.
- Usando `-getmerge` con la opción `-nl`, fusiona `Practicas_HDFS/Textos` en un fichero local `fusion_con_saltos.txt` (la opción `-nl` añade un salto de línea entre ficheros). Compara su contenido con el fusionado del ejemplo.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 8. Copiar y mover dentro de HDFS: `-cp` y `-mv`

Estos comandos trabajan **dentro** del sistema distribuido, sin pasar por el sistema local:

- `-cp`: copia ficheros o directorios. Con `-f` sobrescribe el destino, y con `-p` conserva atributos (permisos, propietario, fechas).
- `-mv`: mueve ficheros o directorios. También sirve para **renombrar**. Es una operación muy barata porque solo modifica los metadatos del namenode, no mueve los bloques de datos.

Fíjate en que los datos no viajan por tu máquina en ningún momento.

In [ ]:
%%bash

# Copia de un fichero a otro directorio
hdfs dfs -cp -f Practicas_HDFS/Textos/tres.txt Practicas_HDFS/Datos/2026

# Renombrado con mv
hdfs dfs -mv Practicas_HDFS/Datos/2026/tres.txt Practicas_HDFS/Datos/2026/tercero.txt

hdfs dfs -ls -R Practicas_HDFS/Datos

**Ejercicio 7:**
- Crea el directorio `Practicas_HDFS/Copia_seguridad`.
- Copia dentro de él, de una sola vez, todos los ficheros `.txt` de `Practicas_HDFS/Textos` (usa un comodín).
- Renombra el fichero `uno.txt` de la copia de seguridad a `uno_backup.txt`.
- Comprueba el resultado con `-ls`.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 9. Espacio ocupado: `-du`, `-df` y `-count`

- `-du`: muestra el espacio que ocupan los ficheros y directorios. Con `-s` da un resumen (un único total) y con `-h` usa unidades legibles. En Hadoop 3 la salida tiene dos columnas de tamaño: la primera es el tamaño real de los datos y la segunda el espacio total consumido **teniendo en cuenta la replicación**.
- `-df`: muestra la capacidad total, el espacio usado y el disponible del sistema de ficheros distribuido.
- `-count`: cuenta directorios, ficheros y bytes bajo una ruta. Con `-q` añade información de cuotas.

In [ ]:
%%bash

echo '--- du de cada elemento ---'
hdfs dfs -du -h Practicas_HDFS
echo
echo '--- du resumen ---'
hdfs dfs -du -s -h Practicas_HDFS
echo
echo '--- df del sistema de ficheros ---'
hdfs dfs -df -h /
echo
echo '--- count: directorios, ficheros y bytes ---'
hdfs dfs -count -h Practicas_HDFS

**Ejercicio 8:**
- Muestra el espacio total ocupado por `Practicas_HDFS/Textos` en formato legible.
- Observa la diferencia entre las dos columnas de tamaño que devuelve `-du`.
- Averigua cuántos directorios y cuántos ficheros hay en `Practicas_HDFS` con `-count`.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


**Escribe aquí por qué las dos columnas de `-du` no coinciden:** 

# 10. Permisos y propietarios: `-chmod`, `-chown` y `-chgrp`

HDFS usa un modelo de permisos similar al de POSIX: lectura (`r`), escritura (`w`) y ejecución (`x`) para el propietario, el grupo y el resto. En un directorio, `x` permite acceder a su contenido. A diferencia de Linux, los ficheros no se ejecutan, por lo que `x` en un fichero carece de efecto.

- `-chmod [-R] <modo> <ruta>`: cambia los permisos, en octal (`750`) o simbólico (`u+x`).
- `-chown [-R] <usuario>[:<grupo>] <ruta>`: cambia propietario (y opcionalmente grupo). Normalmente solo lo puede hacer el superusuario de HDFS.
- `-chgrp [-R] <grupo> <ruta>`: cambia el grupo.

La opción `-R` aplica el cambio de forma recursiva.

In [ ]:
%%bash

echo '--- Antes ---'
hdfs dfs -ls Practicas_HDFS/Textos

hdfs dfs -chmod 640 Practicas_HDFS/Textos/uno.txt
hdfs dfs -chmod -R 750 Practicas_HDFS/Datos

echo
echo '--- Despues ---'
hdfs dfs -ls Practicas_HDFS/Textos
hdfs dfs -ls Practicas_HDFS/Datos

**Ejercicio 9:**
- Deja el fichero `Practicas_HDFS/Textos/dos.txt` con permisos de lectura y escritura solo para el propietario (`600`).
- Quita el permiso de escritura al grupo y al resto sobre todo el directorio `Practicas_HDFS/Copia_seguridad` de forma recursiva, usando la notación simbólica (`go-w`).
- Comprueba los cambios con `-ls`.
- Intenta cambiar el propietario de `dos.txt` con `-chown` a otro usuario. ¿Qué ocurre? Anota por qué.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 11. Bloques y replicación: `-stat`, `-setrep` y `fsck`

Una de las claves de HDFS es que los ficheros se dividen en **bloques** (por defecto de 128 MB) y cada bloque se **replica** en varios datanodes (por defecto, 3 copias).

- `-stat <formato> <ruta>`: muestra información de un fichero. Algunos especificadores de formato: `%n` (nombre), `%b` (tamaño en bytes), `%o` (tamaño de bloque), `%r` (factor de replicación), `%u` (propietario), `%y` (fecha de modificación).
- `-setrep [-R] [-w] <n> <ruta>`: cambia el factor de replicación. Con `-w` espera a que la replicación se complete.
- `hdfs fsck <ruta> -files -blocks -locations`: comando de administración (no es parte de `dfs`) que comprueba la salud del sistema de ficheros y muestra de qué bloques consta cada fichero y en qué datanodes está cada réplica.

In [ ]:
%%bash

hdfs dfs -stat '%n | tamano: %b bytes | bloque: %o bytes | replicas: %r' Practicas_HDFS/Textos/uno.txt

echo
echo '--- Reducimos la replicacion a 2 ---'
hdfs dfs -setrep -w 2 Practicas_HDFS/Textos/uno.txt
hdfs dfs -stat '%n | replicas: %r' Practicas_HDFS/Textos/uno.txt

echo
echo '--- Informe fsck: bloques y ubicacion de las replicas ---'
hdfs fsck Practicas_HDFS/Textos/uno.txt -files -blocks -locations

**Ejercicio 10:**
- Cambia a 1 el factor de replicación de `Practicas_HDFS/Textos/dos.txt` y comprueba con `-stat` que se ha aplicado.
- Ejecuta `hdfs fsck` sobre ese fichero y localiza en la salida cuántas réplicas tiene y en qué datanode está.
- Restablece después la replicación a 3.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


**Escribe tu respuesta aquí:** ¿qué ventaja aporta tener 3 réplicas? ¿Y qué inconveniente? 

# 12. Buscar y comprobar: `-find`, `-test` y `-checksum`

- `-find <ruta> -name <patron>`: busca ficheros por nombre (admite comodines; `-iname` ignora mayúsculas).
- `-test -e|-d|-f|-z <ruta>`: comprueba condiciones y devuelve un código de salida (0 si es cierto): `-e` existe, `-d` es directorio, `-f` es fichero, `-z` tiene tamaño cero. No imprime nada, por lo que se suele usar con `&&` y `||` en scripts.
- `-checksum <ruta>`: muestra la suma de comprobación del fichero, útil para verificar integridad.

In [ ]:
%%bash

echo '--- Todos los .txt bajo Practicas_HDFS ---'
hdfs dfs -find Practicas_HDFS -name '*.txt'
echo

echo '--- Comprobaciones con test ---'
hdfs dfs -test -d Practicas_HDFS/Textos && echo 'Textos es un directorio'
hdfs dfs -test -e Practicas_HDFS/no_existe.txt || echo 'no_existe.txt NO existe'
hdfs dfs -test -z Practicas_HDFS/vacio.txt && echo 'vacio.txt esta vacio'
echo

echo '--- Checksum ---'
hdfs dfs -checksum Practicas_HDFS/Textos/tres.txt

**Ejercicio 11:**
- Busca todos los ficheros cuyo nombre empiece por `uno` bajo `Practicas_HDFS`.
- Escribe una pequeña comprobación que muestre el mensaje `El fichero notas.csv existe` solo si `Practicas_HDFS/Datos/notas.csv` existe, y `Falta notas.csv` en caso contrario (combina `-test`, `&&` y `||`).
- Calcula el checksum de `uno.txt` original y de su copia en `Copia_seguridad`. ¿Son iguales? ¿Por qué sí o por qué no? (pista: recuerda que uno de ellos se modificó con `-appendToFile` antes de copiarse, y que el otro no).

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 13. Borrar datos: `-rm`, `-rmdir` y `-expunge`

- `-rm <fichero>`: borra ficheros. Para borrar directorios y su contenido hay que usar `-r`. Con `-f` no da error si la ruta no existe.
- `-rmdir <directorio>`: borra únicamente directorios **vacíos**.
- `-skipTrash`: borra definitivamente, sin pasar por la papelera. Si la papelera de HDFS está activada (propiedad `fs.trash.interval`), un borrado normal mueve los datos a `.Trash` en tu directorio personal, desde donde se pueden recuperar durante un tiempo.
- `-expunge`: vacía la papelera.

**Ten cuidado:** con `-r -skipTrash` no hay vuelta atrás. Antes de borrar un directorio, comprueba con `-ls` qué contiene.

In [ ]:
%%bash

# Preparamos algo para borrar
hdfs dfs -mkdir -p Practicas_HDFS/Temporal/Vacio
hdfs dfs -put -f Datos_locales/extra.txt Practicas_HDFS/Temporal
hdfs dfs -ls -R Practicas_HDFS/Temporal
echo

# Borrado de un fichero
hdfs dfs -rm Practicas_HDFS/Temporal/extra.txt

# Borrado de un directorio vacio
hdfs dfs -rmdir Practicas_HDFS/Temporal/Vacio

# Borrado de un directorio (ya vacio) con rm -r
hdfs dfs -rm -r Practicas_HDFS/Temporal
echo
hdfs dfs -ls Practicas_HDFS

**Ejercicio 12:**
- Crea el directorio `Practicas_HDFS/Basura` con un subdirectorio llamado `Subcarpeta` y sube a `Basura` el fichero `Datos_locales/extra.txt`.
- Intenta borrar `Basura` con `-rmdir` y observa el error. ¿Por qué falla?
- Bórralo ahora correctamente con `-rm -r`, indicando `-skipTrash`.
- Comprueba con `-ls` que ya no existe.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 14. Ejercicio final integrador

**Ejercicio 13:** pon en práctica todo lo aprendido. En un único bloque, y comentando cada paso con `#`:

1. Crea en HDFS la estructura `Proyecto_Final/entrada` y `Proyecto_Final/salida`.
2. Sube a `entrada` los tres ficheros `uno.txt`, `dos.txt` y `tres.txt` de `Datos_locales`, con una única orden.
3. Copia todo el contenido de `entrada` a `salida`.
4. Establece un factor de replicación de 2 para todos los ficheros de `salida`, de forma recursiva.
5. Comprueba con `-stat` la replicación de alguno de ellos.
6. Fusiona los ficheros de `entrada` en un único fichero local llamado `Datos_locales/resultado_final.txt`.
7. Muestra el espacio ocupado por `Proyecto_Final` en formato legible y con resumen.
8. Elimina por completo `Proyecto_Final` de HDFS y verifica que ya no existe con `-test`.

In [ ]:
%%bash
##Realiza el ejercicio en este bloque:


# 15. Limpieza

Cuando termines, es buena práctica dejar el sistema como lo encontraste. El siguiente bloque elimina todo lo creado en este cuaderno, tanto en HDFS como en el sistema de ficheros local. Si has creado más cosas por tu cuenta, límpialas también.

In [ ]:
%%bash

hdfs dfs -rm -r -f -skipTrash Practicas_HDFS Proyecto Proyecto_Final
rm -rf Datos_locales Descargas_HDFS Descargas_Ejercicio6

echo '--- HDFS tras la limpieza ---'
hdfs dfs -ls

# Resumen de comandos

| Tarea | Comando HDFS | Equivalente Linux |
|---|---|---|
| Listar | `hdfs dfs -ls [-h] [-R] ruta` | `ls` |
| Crear directorio | `hdfs dfs -mkdir [-p] ruta` | `mkdir` |
| Crear fichero vacío | `hdfs dfs -touchz ruta` | `touch` |
| Subir (local a HDFS) | `hdfs dfs -put [-f] origen destino` | `cp` |
| Subir y borrar local | `hdfs dfs -moveFromLocal origen destino` | `mv` |
| Añadir al final | `hdfs dfs -appendToFile local fichero_hdfs` | `cat local >> fichero` |
| Ver contenido | `hdfs dfs -cat ruta` | `cat` |
| Ver inicio / final | `hdfs dfs -head ruta` / `-tail ruta` | `head` / `tail` |
| Descargar (HDFS a local) | `hdfs dfs -get [-f] origen destino` | `cp` |
| Descargar y fusionar | `hdfs dfs -getmerge dir_hdfs fichero_local` | `cat dir/* > fichero` |
| Copiar dentro de HDFS | `hdfs dfs -cp [-f] origen destino` | `cp` |
| Mover / renombrar | `hdfs dfs -mv origen destino` | `mv` |
| Espacio ocupado | `hdfs dfs -du [-s] [-h] ruta` | `du` |
| Espacio libre | `hdfs dfs -df [-h] /` | `df` |
| Contar elementos | `hdfs dfs -count [-h] ruta` | `find \| wc -l` (aprox.) |
| Permisos | `hdfs dfs -chmod [-R] modo ruta` | `chmod` |
| Propietario / grupo | `hdfs dfs -chown` / `-chgrp` | `chown` / `chgrp` |
| Información de fichero | `hdfs dfs -stat formato ruta` | `stat` |
| Cambiar replicación | `hdfs dfs -setrep [-w] n ruta` | (propio de HDFS) |
| Buscar | `hdfs dfs -find ruta -name patron` | `find` |
| Comprobar condición | `hdfs dfs -test -e\|-d\|-f\|-z ruta` | `test` |
| Borrar fichero / árbol | `hdfs dfs -rm [-r] [-f] [-skipTrash] ruta` | `rm` |
| Borrar directorio vacío | `hdfs dfs -rmdir ruta` | `rmdir` |
| Vaciar papelera | `hdfs dfs -expunge` | (propio de HDFS) |